<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/13_trajectory_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13 · Testing the path, not just the answer

Both evaluators in lesson 12 read the final message. That is enough for a chatbot and not enough
for an agent, because an agent can be right by accident.

An agent that answers "repair only" because it read the policy is working. An agent that answers
"repair only" because 62 days sounds like a lot is a coin flip that happened to land well — and it
will land badly on an example you have not written yet.

**New in this lesson:** trajectories, extracting tool calls from a run, path assertions as stored
code evaluators, trajectory LLM judges, and `agentevals`

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq --progress-bar off \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "agentevals~=0.0.9"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-13-trajectory-evals"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

In [ ]:
#@title Connect to the shared deployment (run me) { display-mode: "form" }
# --- snippet:remote_agent v1 ---
import hashlib

import httpx
from langgraph.pregel.remote import RemoteGraph

HOST_API = "https://api.host.langchain.com"
LS_API = "https://api.smith.langchain.com/api/v1"
HEADERS = {"x-api-key": key}


def find_deployment(name: str = "support-agent") -> dict:
    """The shared deployment's record, looked up by name."""
    response = httpx.get(
        f"{HOST_API}/v2/deployments",
        params={"name_contains": name},
        headers={"X-Api-Key": os.environ["LANGSMITH_API_KEY"]},
        timeout=30,
    )
    response.raise_for_status()
    for record in response.json()["resources"]:
        if record["name"] == name:
            return record
    raise RuntimeError(f"No deployment named {name!r} in this workspace.")


DEPLOYMENT = find_deployment()

# Every attendee's calls land in this one project. That is the point: shared traffic.
TRAFFIC_PROJECT_ID = DEPLOYMENT["tracer_session_id"]

# "support" is the graph key from langgraph.json in lesson 10.
support = RemoteGraph("support", url=DEPLOYMENT["url"], api_key=key)


def last_text(result: dict) -> str:
    """The final reply. A deployment returns JSON, so messages are dicts."""
    content = result["messages"][-1].get("content") or ""
    if isinstance(content, list):
        # The model returns reasoning blocks alongside the answer; keep the answer.
        return " ".join(part["text"] for part in content
                        if isinstance(part, dict) and part.get("type") == "text")
    return str(content)


def tool_names(result: dict) -> list[str]:
    """Every tool the run called, in order."""
    return [call["name"]
            for message in result["messages"]
            for call in (message.get("tool_calls") or [])]


# Everyone shares the agent; nobody shares your datasets. Derived from your key so
# it is unique to you and the same every time you run this.
ME = hashlib.sha256(key.encode()).hexdigest()[:8]
# --- /snippet ---

print(f"{DEPLOYMENT['name']}: {DEPLOYMENT['status']} | you are {ME}")

In [ ]:
from langsmith import Client

client = Client()

---

## 1. What the agent actually did

A **trajectory** is the ordered sequence of steps a run took — which tools it called, with what
arguments, in what order. It is already in the trace; you just have to read it.

In [ ]:
result = support.invoke({"messages": [{"role": "user", "content":
    "Ticket T-6: the laptop stand on order 1047 wobbles. What can we offer?"
}]})

for message in result["messages"]:
    kind = message.get("type", "?")
    calls = message.get("tool_calls") or []
    if calls:
        for call in calls:
            print(f"  {kind:10} -> {call['name']}({call['args']})")
    else:
        content = message.get("content") or ""
        if isinstance(content, list):
            content = " ".join(p.get("text", "") for p in content if isinstance(p, dict))
        print(f"  {kind:10}    {str(content)[:80]}")

That list is the evidence. Somewhere in it the agent either loaded the refund-decision skill and
looked up order 1047, or it did not.

Two failure modes hide from an answer-only evaluator:

- **The lucky guess.** Right answer, no lookup. Passes every output check you have.
- **The expensive detour.** Right answer after eleven tool calls, three of them repeats. Passes
  every output check you have, and costs you five times as much.

Neither is visible in the final message, which is exactly why trajectory evals exist.

---

## 2. A path assertion, stored in LangSmith

The same `perform_eval` contract from lesson 12, reading `outputs.messages` instead of just the
last one. No new machinery — a different question asked of the same data.

In [ ]:
PATH_CHECK = r"""
def perform_eval(run, example):
    # Did the agent look up the order before deciding?
    messages = (run["outputs"] or {}).get("messages") or []

    sequence = []
    for message in messages:
        for call in message.get("tool_calls") or []:
            name = call.get("name")
            if name:
                sequence.append(name)

    looked_up = "lookup_order" in sequence
    decided_at = len(sequence)
    for index, name in enumerate(sequence):
        if name == "lookup_order":
            decided_at = index
            break

    return {
        "looked_up_order": 1 if looked_up else 0,
        "read_policy": 1 if any("skill" in n or "read_file" in n for n in sequence) else 0,
        "tool_calls": len(sequence),
        "repeated_calls": len(sequence) - len(set(sequence)),
    }
"""

evaluator = httpx.post(
    f"{LS_API}/platform/evaluators",
    headers=HEADERS,
    json={
        "name": f"path-check-{ME}",
        "type": "code",
        "code_evaluator": {"code": PATH_CHECK, "language": "python"},
    },
    timeout=60,
).json()["evaluator"]

print("id:", evaluator["id"])
print("feedback keys:", evaluator["feedback_keys"])

Four metrics, none of which look at the answer:

- `looked_up_order` — did it check the facts, or guess?
- `read_policy` — did it load the procedure that governs this decision?
- `tool_calls` — a cost proxy you can chart per commit.
- `repeated_calls` — the same tool twice with the same effect, which is usually a confused agent.

`tool_calls` deserves a note. It is not a pass/fail, and treating it as one would be a mistake —
there is no correct number. It is a **number to watch**: when a prompt change quietly doubles it,
you find out from a chart instead of from a bill.

In [ ]:
# Same local test as before. Path logic is fiddly enough to be worth checking by hand.
namespace = {}
exec(PATH_CHECK, namespace)

guessed = {"outputs": {"messages": [
    {"content": "That would be a repair."},
]}}
checked = {"outputs": {"messages": [
    {"tool_calls": [{"name": "read_file", "args": {"path": "/skills/refund-decision/SKILL.md"}}]},
    {"tool_calls": [{"name": "lookup_order", "args": {"order_id": "1047"}}]},
    {"content": "Delivered 62 days ago, so repair only."},
]}}

print("guessed:", namespace["perform_eval"](guessed, {}))
print("checked:", namespace["perform_eval"](checked, {}))

Same final answer in both. Completely different scores. That gap is the whole reason this lesson
exists.

---

## 3. Score the path across a dataset

Attach it the same way as any other evaluator and run the experiment.

In [ ]:
name = f"refund-decisions-{ME}"
dataset = (client.read_dataset(dataset_name=name)
           if client.has_dataset(dataset_name=name)
           else client.create_dataset(name))

httpx.post(
    f"{LS_API}/runs/rules",
    headers=HEADERS,
    json={
        "display_name": f"score-path-{ME}",
        "dataset_id": str(dataset.id),
        "evaluator_id": evaluator["id"],
        "sampling_rate": 1.0,
        "is_enabled": True,
    },
    timeout=30,
)

results = client.evaluate(
    support,
    data=dataset.name,
    experiment_prefix=f"trajectory-{ME}",
    max_concurrency=2,
    evaluators=[],
)
print(results.experiment_name)

In the experiment view, sort by `looked_up_order`. The rows where it is 0 are the ones to read —
whatever the agent answered, it answered without checking.

---

## 4. When the path is too varied to assert

"Called `lookup_order` at least once" is a fact. "Went about this sensibly" is not — and plenty of
real requirements are the second kind. A ticket might legitimately be answered in three tool calls
or seven, depending on what the first one returned.

For those, judge the trajectory with a model. `agentevals` does this locally, which makes it the
easiest way to see what such a rubric looks like before you commit to storing one.

In [ ]:
from agentevals.trajectory.llm import create_trajectory_llm_as_judge

trajectory_judge = create_trajectory_llm_as_judge(
    model=MODEL,
    prompt=(
        "You are grading whether a support agent investigated before deciding.\n\n"
        "A good trajectory establishes the facts it needs (the order's age and status) and consults "
        "the refund policy before committing to an outcome. Extra exploration is fine. Deciding "
        "first and justifying afterwards is not.\n\n"
        "Trajectory:\n{outputs}\n\n"
        "Grade it."
    ),
)

verdict = trajectory_judge(outputs=result["messages"])
print(verdict)

The same rubric can be stored in LangSmith rather than run here — a rule takes
`trajectory_evaluators` alongside the `evaluators` you have been using, holding a prompt and a
schema exactly like lesson 12's judge. The choice is the same one as before: locally while you are
still figuring out the rubric, stored once you want it applied to traffic you are not watching.

---

## 5. Matching against a known-good path

Sometimes you do know the exact path, because a colleague already produced a good run and you want
to keep it that way. `agentevals` compares two trajectories directly.

In [ ]:
from agentevals.trajectory.match import create_trajectory_match_evaluator

reference = [
    {"role": "user", "content": "Ticket T-6: the laptop stand on order 1047 wobbles."},
    {"role": "assistant", "content": "", "tool_calls": [
        {"id": "1", "function": {"name": "lookup_order", "arguments": '{"order_id": "1047"}'}}]},
    {"role": "tool", "content": "Order 1047: delivered 62 days ago", "tool_call_id": "1"},
    {"role": "assistant", "content": "Repair only."},
]

for mode in ["strict", "unordered", "subset", "superset"]:
    evaluator = create_trajectory_match_evaluator(
        trajectory_match_mode=mode,
        tool_args_match_mode="ignore",
    )
    print(f"  {mode:10}", evaluator(outputs=reference, reference_outputs=reference)["score"])

The four modes are a spectrum from brittle to permissive, and picking the wrong one is the most
common way a trajectory suite becomes useless:

| Mode | Passes when | Use it when |
|---|---|---|
| `strict` | same tools, same order | the sequence is the requirement (auth before charge) |
| `unordered` | same tools, any order | order genuinely does not matter |
| `superset` | the run did **at least** the reference steps | "must look up the order" — extra steps allowed |
| `superset` + `tool_args_match_mode="ignore"` | as above, arguments unchecked | you care that it looked, not what it looked at |
| `subset` | the run did **no more than** the reference | tightening a wasteful agent |

`strict` looks like the rigorous choice and is usually the wrong one. Agents reorder harmless calls
between model versions, so a strict suite fails on changes that are not regressions, and a suite
that cries wolf gets muted. Start at `superset`: assert the steps that must happen, stay silent
about the rest.

---

## 6. What is worth asserting

The temptation with trajectory evals is to pin everything, which produces a suite that breaks every
time anyone touches a prompt. A useful rule of thumb: assert the steps that exist for a **reason
outside the model**.

Worth asserting:

- **Facts before commitments.** Looked up the order before promising a refund.
- **Policy before judgement.** Read the procedure that governs the decision.
- **Order where order is the requirement.** Verify identity before disclosing account details.
- **Absence.** Did *not* call `issue_refund` on a $429 order without approval. Negative assertions
  age far better than positive ones.
- **Bounded effort.** Under N tool calls, or no tool called three times in a row.

Not worth asserting:

- The exact number of calls on a happy path.
- The order of two independent lookups.
- Which of two equivalent tools it picked.
- Anything you would happily change yourself next week.

The test: **if this assertion failed, would you investigate the agent, or edit the test?** If the
honest answer is "edit the test", it was never a test.

---

## 📌 Key takeaways

- A trajectory is the ordered sequence of tool calls and arguments — already in the trace, waiting to be read.
- Answer-only evaluators cannot see the lucky guess or the expensive detour.
- A path assertion is the same `perform_eval` contract, reading all of `outputs.messages` instead of the last one.
- Useful path metrics include cost proxies (`tool_calls`) that are numbers to watch, not thresholds to pass.
- The same answer via two different paths should score differently; that gap is the point.
- Use an LLM trajectory judge when "did this go sensibly?" has no assertable form.
- `agentevals` match modes run brittle to permissive: `strict`, `unordered`, `superset`, `subset`.
- `strict` fails on harmless reorderings, so most suites should start at `superset`.
- Assert steps that exist for reasons outside the model; skip anything you would cheerfully change next week.
- **Negative** assertions (it must not do X) age better than positive ones.
- An assertion that has never failed is untested. Break it on purpose once.

---

## ➡️ Next

**[14 · Online evals and closing the loop](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/14_online_evals.ipynb)**

Everything so far has been offline: a dataset you chose, run when you asked. Next, the same
evaluators pointed at live traffic — and the loop that uses human corrections to make a judge you
can trust.